# 06 · Worker energy model and local refinement

**Status: UNEXECUTED FIRST DRAFT.** This notebook contains working implementations to inspect and refine when run. No trained capability or benchmark result is asserted.

**Purpose:** Generate legal action/coordinate proposals with a conditional energy model and refine their semantic targets under a finite budget.

**Inputs:** Notebook 05 geometry and budget utilities; context and goal latents; public action masks.

**Outputs:** ConditionalEnergy, ProposalHeads and WorkerEBM with differentiable training path and bounded inference proposal path.

**Expected runtime:** Under a minute for meta parameter checks; full numerical proposal test is in 09. These are planning estimates; measure on this instance.

**Checkpoint directory:** `checkpoints/<run_id>/` under the project root. Definitions in this notebook are included in every source manifest.

**Exact resume:** Restart the AHR v2 XL kernel and Run All. This notebook defines components and runs bounded checks; it does not start a long training job. Training continuation is handled in 10 and 11 from the latest valid checkpoint.

The configuration remains the full XL design (`d_model=1024` by default). Any explicitly identified synthetic fixture verifies plumbing only and is not a smaller AHR training configuration.


## Load explicit notebook dependencies
Only tagged definitions are loaded; dependency execution cells are not run.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([5], globals())

## Conditional energy and proposal networks

An energy score is not a generator or a truth verifier. Explicit proposal heads generate candidate latents, actions and click distributions. The energy network ranks them; gradients refine only candidate latents during planning.

In [ ]:
def broadcast_semantics(context, candidate, goal):
    while context.ndim < candidate.ndim:
        context = context.unsqueeze(-2)
    while goal.ndim < candidate.ndim:
        goal = goal.unsqueeze(-2)
    context, candidate, goal = torch.broadcast_tensors(context, candidate, goal)
    return context, candidate, goal

class ConditionalEnergy(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        m = cfg['model']
        width, hidden = int(m['d_model']), int(m['energy_hidden'])
        layers = [nn.LayerNorm(3*width), nn.Linear(3*width, hidden), nn.GELU()]
        for _ in range(max(0, int(m['energy_layers'])-1)):
            layers += [nn.Linear(hidden, hidden), nn.GELU()]
        layers.append(nn.Linear(hidden, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, context, candidate, goal):
        context, candidate, goal = broadcast_semantics(context, candidate, goal)
        learned = self.network(torch.cat((context, candidate, goal), dim=-1)).squeeze(-1)
        alignment = (candidate.float()-goal.float()).square().mean(dim=-1)
        return learned.float() + 0.05*alignment

class ProposalHeads(nn.Module):
    def __init__(self, cfg, maximum_candidates=128):
        super().__init__()
        m = cfg['model']
        width = int(m['d_model'])
        self.maximum_candidates = int(maximum_candidates)
        self.latent_clip = float(cfg['reasoning']['latent_clip'])
        self.base = nn.Sequential(nn.LayerNorm(2*width), nn.Linear(2*width, width), nn.GELU(), nn.Linear(width, width))
        self.register_buffer('offsets', torch.randn(self.maximum_candidates, width)*0.02)
        self.action_head = nn.Linear(width, int(m['action_dim']))
        self.click_head = nn.Linear(width, 2*int(m['grid_size']))
        self.grid_size = int(m['grid_size'])

    def direct(self, context, goal):
        latent = (context + self.base(torch.cat((context, goal), dim=-1))).clamp(-self.latent_clip, self.latent_clip)
        return {'latent': latent, 'action_logits': self.action_head(latent),
                'click_logits': self.click_head(latent).reshape(*latent.shape[:-1], 2, self.grid_size)}

    def candidates(self, context, goal, count, noise_scale=0.05):
        count = int(count)
        if not 1 <= count <= self.maximum_candidates:
            raise ValueError(f'Candidate count must be 1..{self.maximum_candidates}.')
        base = self.direct(context, goal)['latent'].unsqueeze(1)
        noise = torch.randn(context.shape[0], count, context.shape[-1], device=context.device, dtype=context.dtype)*noise_scale
        return (base + self.offsets[:count].unsqueeze(0) + noise).clamp(-self.latent_clip, self.latent_clip)

    def decode(self, candidates, mask):
        logits = mask_logits(self.action_head(candidates), mask)
        batch, count, actions = logits.shape
        selected_actions = torch.multinomial(logits.float().reshape(-1, actions).softmax(-1), 1).reshape(batch, count)
        click_logits = self.click_head(candidates).reshape(batch, count, 2, self.grid_size)
        pixels = torch.multinomial(click_logits.float().reshape(-1, self.grid_size).softmax(-1), 1).reshape(batch, count, 2)
        xy = pixels.to(candidates.dtype) * (2.0/(self.grid_size-1)) - 1.0
        xy = torch.where((selected_actions == 6).unsqueeze(-1), xy, torch.zeros_like(xy))
        return selected_actions, xy, logits, click_logits

## Worker proposal and bounded energy descent

Refinement detaches between steps, uses FP32 latent gradients and never creates an unbounded unrolled training graph. `.direct` remains differentiable for teacher, policy and dynamics losses. The worker favors local changes; Scout will use wider proposals.

In [ ]:
class WorkerEBM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = copy.deepcopy(cfg)
        self.energy_network = ConditionalEnergy(cfg)
        self.proposal_heads = ProposalHeads(cfg)
        self.refine_steps = int(cfg['reasoning']['refine_steps'])
        self.latent_clip = float(cfg['reasoning']['latent_clip'])
        self.refine_rate = 0.15

    def energy(self, context, candidate, goal):
        return self.energy_network(context, candidate, goal)

    def direct(self, context, goal):
        return self.proposal_heads.direct(context, goal)

    def propose(self, context, goal, mask, count):
        validate_action_mask(mask, context.shape[0])
        with torch.no_grad():
            candidates = self.proposal_heads.candidates(context, goal, count, noise_scale=0.05)
        for _ in range(self.refine_steps):
            with torch.enable_grad(), torch.autocast(device_type=context.device.type, enabled=False):
                variable = candidates.detach().float().requires_grad_(True)
                objective = self.energy(context.detach().float(), variable, goal.detach().float()).sum()
                gradient = torch.autograd.grad(objective, variable, create_graph=False)[0]
                gradient = gradient / gradient.norm(dim=-1, keepdim=True).clamp_min(1.0)
                candidates = (variable-self.refine_rate*gradient).clamp(-self.latent_clip, self.latent_clip).detach()
        with torch.no_grad():
            candidates = candidates.to(context.dtype)
            actions, xy, logits, click_logits = self.proposal_heads.decode(candidates, mask)
            energies = self.energy(context, candidates, goal)
        return {'latents': candidates, 'actions': actions, 'xy': xy, 'energies': energies,
                'action_logits': logits, 'click_logits': click_logits}

## Run All: bounded validation

Metadata checks allocate no real XL weights. Notebook 09 exercises actual Worker inference and gradient flow on the full model.

In [ ]:
cfg = load_config()
with torch.device('meta'):
    worker_metadata = WorkerEBM(cfg)
print({'worker_parameters': sum(p.numel() for p in worker_metadata.parameters()),
       'refinement_steps': cfg['reasoning']['refine_steps'],
       'candidate_count': cfg['reasoning']['worker_candidates'],
       'energy_is_verification': False})
del worker_metadata
assert cfg['reasoning']['worker_candidates'] <= 128
assert 0 <= cfg['reasoning']['refine_steps'] <= 64